[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Реализуйте **multi-head cross-attention**. Query-последовательность решает, какую информацию извлечь, а отдельная key/value-последовательность предоставляет память. Типичный пример: decoder tokens обращаются ко всем encoder states.

### Формы по шагам
При `D=d_model`, `H=num_heads`, `d_h=D/H` требуется `D % H == 0`.

- projected `Q`: `(B, H, S_q, d_h)`;
- projected `K,V`: `(B, H, S_kv, d_h)`;
- attention scores `QKᵀ / sqrt(d_h)`: `(B, H, S_q, S_kv)`;
- context после умножения на `V`: `(B, H, S_q, d_h)`;
- concat heads и output projection возвращают `(B, S_q, D)`.

Softmax применяется по последней размерности `S_kv`: для каждой query и каждого head веса всех source positions суммируются в единицу. Масштаб `sqrt(d_h)` удерживает logits attention в разумном диапазоне.

### Контракт
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Отличия от self-attention
- Q строится из `x_q`, а K и V — из `x_kv`;
- длины `S_q` и `S_kv` могут различаться;
- causal mask в этом упражнении нет: каждая query видит все KV positions.

### План реализации
1. Создайте четыре Linear-проекции: Q, K, V и output.
2. Разделите последнюю dimension на heads и переставьте axes.
3. Вычислите scaled dot-product scores и softmax по KV positions.
4. Получите weighted sum values.
5. Верните heads в исходный порядок, объедините и примените output projection.

### Быстрые самопроверки
- output length всегда равна `S_q`, не `S_kv`;
- работает при `S_q != S_kv`;
- изменение последней KV position влияет даже на первую query;
- gradients доходят и до `x_q`, и до `x_kv`;
- attention weights суммируются в единицу по `S_kv`.

### Типичные ошибки
- K/V случайно строятся из `x_q`;
- scaling на `sqrt(D)` вместо `sqrt(d_h)`;
- softmax по query axis;
- transpose последних dimensions K выполнен до разделения heads неверно;
- concat heads нарушает порядок token и head axes.

### Официальные материалы и примеры
- [`torch.nn.MultiheadAttention`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) — формула, shapes query/key/value и cross-attention пример;
- [`scaled_dot_product_attention`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) — эталон операции одного attention-блока.

### Вопросы для самопроверки
1. Почему длину output определяет query sequence?
2. По какой оси должен выполняться Softmax и почему?
3. Как несколько heads позволяют изучать разные типы соответствий?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        pass  # Q from x_q, K/V from x_kv, no causal mask

In [ ]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')